In [ ]:
import os
import subprocess
import sys
import torch

print("=== Deepfake GPU Worker ===")
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError("Kaggle GPU is not enabled.")

print("GPU:", torch.cuda.get_device_name(0))

# Clone/update project
repo = "/kaggle/working/deepfake"
if not os.path.exists(repo):
    subprocess.run([
        "git", "clone",
        "https://github.com/aiexpert807567-dotcom/deepfake.git",
        repo
    ], check=True)
else:
    subprocess.run(["git", "-C", repo, "pull"], check=False)

os.chdir(repo)

# Install worker dependencies
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "-r",
    "worker/requirements.txt"
], check=True)

# Install FFmpeg
subprocess.run([
    "apt-get", "update", "-qq"
], check=False)

subprocess.run([
    "apt-get", "install", "-y", "-qq", "ffmpeg"
], check=False)

# Worker configuration
os.environ["API_URL"] = "https://ai-face-studio-backend-d56h.onrender.com"
os.environ["WORKER_AUTH_TOKEN"] = "studio_worker_secret_token_2026"
os.environ["WORKER_ID"] = "kaggle_t4_gpu"

# Verify CUDA + FFmpeg
import shutil
print("FFmpeg:", shutil.which("ffmpeg"))
print("PyTorch CUDA:", torch.version.cuda)

# Start worker
subprocess.run([sys.executable, "-u", "worker/worker.py"], check=True)
